# 03B — Balanced Topic Modeling (YT aggregation + Rappler chunking)

In [ ]:

# Project paths
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
FIG  = ROOT / "figures"
PROC.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("RAW :", RAW)
print("PROC:", PROC)
print("FIG :", FIG)


In [ ]:
# ---------- Load, Balance, and Model ----------
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import numpy as np

# --- Load Stopwords from notebook 03 ---
def read_lines(path):
    p = ROOT / path
    if not p.exists(): p = Path(path)
    if not p.exists(): return []
    with open(p, "r", encoding="utf-8", errors="ignore") as f:
        return [ln.strip().lower() for ln in f if ln.strip() and not ln.strip().startswith("#")]
basic_sw = set(read_lines("configs/Basic Stopwords.txt"))
domain_sw = set(read_lines("configs/Domain-Specific Stopwords.txt"))
tagalog_sw = set(read_lines("configs/tagalog_stopwords.txt"))

STOPWORDS = list(basic_sw | domain_sw | tagalog_sw)
print(f"Loaded {len(STOPWORDS)} stopwords.")

# --- Load pre-cleaned docs from Notebook 03 ---
docs_path = PROC / "docs_agg_map.csv" 
if not docs_path.exists():
    raise FileNotFoundError("Could not find 'docs_agg_map.csv'. Please re-run Notebook 03 to create it.")

docs = pd.read_csv(docs_path)
docs['text'] = docs['text'].fillna('').astype(str)

# Balance by sampling
sizes = docs["platform"].value_counts()
n = sizes.min()  # Cap to the smallest platform (e.g., 37 docs)
if n < 1: n = sizes.loc[sizes > 0].min() # Ensure at least 1 if any platform has docs
    
balanced = (docs
            .groupby("platform", group_keys=False)
            .apply(lambda g: g.sample(min(len(g), n), random_state=42))
            .reset_index(drop=True))

print("Original counts:", sizes.to_dict())
print("Balanced counts:", balanced["platform"].value_counts().to_dict())

# Vectorize
cv = CountVectorizer(
    stop_words=STOPWORDS,
    ngram_range=(1, 2),  # looks for 1-word and 2-word phrases
    min_df=10,           # must appear in at least 10 documents
    max_df=0.85          # ignore words that are in > 85% of all documents
)

X = cv.fit_transform(balanced["text"].tolist())
vocab = cv.get_feature_names_out()
print(f"Matrix shape: {X.shape}")

# --- Latent Dirichlet Allocation ---
lda = LatentDirichletAllocation(n_components=7, random_state=42, learning_method="batch")
lda.fit(X)

# --- Print all the top terms ---
def top_terms_print(model, feat, n=12):
    comps = model.components_
    for t, row in enumerate(comps):
        terms = np.array(feat)[row.argsort()[-n:][::-1]]
        print(f"Topic {t}: {', '.join(terms)}")
top_terms_print(lda, vocab)

# Save matrices with '_balanced' suffix
pd.DataFrame(lda.components_, columns=vocab).to_csv(PROC / "topic_term_matrix_balanced.csv", index=False)
pd.DataFrame(lda.transform(X)).to_csv(PROC / "doc_topic_matrix_balanced.csv", index=False)
balanced.to_csv(PROC / "docs_balanced_index.csv", index=False)
print("Saved balanced matrices.")

In [ ]:
# ---------- Top Terms plus Bar Charts (Balanced) ----------
import matplotlib.pyplot as plt
import numpy as np

def top_terms_plot(model, vocab, n=12):
    out = []
    for t, row in enumerate(model.components_):
        idx = row.argsort()[-n:][::-1]
        terms = [vocab[i] for i in idx]
        weights = [row[i] for i in idx]
        out.append((t, terms, weights))
    return out

tops = top_terms_plot(lda, vocab, n=12)

with open(PROC / "topic_top_terms_balanced.txt", "w", encoding="utf-8") as f:
    for t, terms, _ in tops:
        f.write(f"Topic {t}: " + ", ".join(terms) + "\n")

for t, terms, weights in tops:
    plt.figure(figsize=(8,4))
    # create a list of colors from a colormap
    colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(terms)))
    plt.barh(terms[::-1], weights[::-1], color=colors)
    plt.xlabel("weight"); plt.title(f"Topic {t} — top terms"); plt.tight_layout()
    plt.savefig(FIG / f"topic_terms_balanced_T{t}.png", dpi=160); plt.close()

print("Saved:", PROC / "topic_top_terms_balanced.txt", "and topic_terms_balanced_T*.png")

In [ ]:
# calculate the topic strengths
strengths_series = pd.DataFrame(lda.components_, columns=vocab).sum(axis=1)

# 1. Give the series a name
strengths_series.name = "strength"

# 2. then save it with the header
strengths_series.to_csv(PROC / "topic_strengths.csv", index=False, header=True)

print("Saved topic_strengths.csv (with 'strength' header)")

In [ ]:
import pandas as pd
import numpy as np

# helper function
def top_terms_df(model, vocab, n=15):
    rows = []
    for t, row in enumerate(model.components_):
        idx = row.argsort()[-n:][::-1]
        terms = [vocab[i] for i in idx]
        weights = [row[i] for i in idx]
        for rank, (term, w) in enumerate(zip(terms, weights), start=1):
            rows.append({"topic": t, "rank": rank, "term": term, "weight": w})
    return pd.DataFrame(rows)

# next set of code is for the setup of the balanced notebook
df_terms_bal = top_terms_df(lda, vocab, n=15) 

df_terms_bal.to_csv(PROC / "topic_top_terms_balanced.csv", index=False)

print("Saved: topic_top_terms_balanced.csv")
print(df_terms_bal.head())